# Official Replay Bad-Interval Feature Audit v1

This diagnostic starts from the official replay gate flip and asks which qlib feature groups were abnormal inside the official replay drawdown intervals. It does not train or tune thresholds. The output is a mechanism-routing artifact for the next experiment.

External basis: Qlib/FinRL style evaluation uses replayed trading paths, while backtest-overfitting work warns against optimizing from source-only winners. Here the only trusted failure path is the official replay daily path, so feature attribution is restricted to that path's worst intervals.

In [1]:
from __future__ import annotations

import json
import math
import os
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / "AGENTS.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "AGENTS.md").exists():
    ROOT = Path("/Users/rziiiii/.openclaw/workspace")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.path_diagnostics import (
    compare_interval_feature_deltas,
    summarize_interval_feature_groups,
)
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.data import build_dataset, flatten_feature_frame, init_qlib
from src.soxl_quant.settings import QLIB_DATA_PATH

patch_gym_compat()

EXPERIMENT = "official_replay_bad_interval_feature_audit_v1"
CARD = ROOT / "runtime/model_selection/card" / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
SOURCE_METRIC_CARD = ROOT / "runtime/model_selection/card/metric_alignment_gate_flip_audit_v1"
SOURCE_REPLAY_CARD = ROOT / "runtime/model_selection/card/short_volshock_bc_initialization_official_replay_v1"
PROFILE_NAME = "sparse_release_bc_ppo_screen_v1"
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0

WALK_FORWARD_WINDOWS = {
    "wf_2022": {
        "train": ("2010-03-11", "2020-12-31"),
        "valid": ("2021-01-01", "2021-12-31"),
        "test": ("2022-01-01", "2022-12-31"),
    },
    "wf_2023_2024": {
        "train": ("2010-03-11", "2022-12-31"),
        "valid": ("2023-01-01", "2023-12-31"),
        "test": ("2024-01-01", "2024-12-31"),
    },
}

EXTERNAL_SOURCES = [
    "https://qlib.readthedocs.io/en/latest/component/report.html",
    "https://github.com/microsoft/qlib",
    "https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2326253",
    "https://arxiv.org/abs/2009.11189",
    "https://arxiv.org/abs/2011.09607",
]


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if hasattr(value, "item"):
        return json_safe(value.item())
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8")


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))

metric_selection = load_json(SOURCE_METRIC_CARD / "selection.json")
metric_summary = load_json(SOURCE_METRIC_CARD / "summary.json")
source_audit = pd.read_csv(SOURCE_METRIC_CARD / "metric_alignment_audit.csv")
source_intervals = pd.read_csv(SOURCE_METRIC_CARD / "worst_official_drawdown_intervals.csv")
source_pk = pd.read_csv(SOURCE_REPLAY_CARD / "source_vs_official_pk.csv")
profile = load_profile(PROFILE_NAME)
print({
    "experiment": EXPERIMENT,
    "metric_source_decision": metric_selection.get("decision"),
    "interval_rows": int(len(source_intervals)),
    "profile": PROFILE_NAME,
})
display(source_audit)
display(source_intervals)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'experiment': 'official_replay_bad_interval_feature_audit_v1', 'metric_source_decision': 'metric_alignment_gate_flip_blocks_training', 'interval_rows': 4, 'profile': 'sparse_release_bc_ppo_screen_v1'}


,candidate,fold,source_relative_return,official_relative_return,source_max_dd,official_max_dd,source_gate_pass,official_gate_pass,source_optimistic_gate_flip,relative_delta_official_minus_source,relative_overstatement_pp,maxdd_delta_official_minus_source,maxdd_understatement_pp,material_relative_overstatement,material_maxdd_understatement,primary_mismatch,required_action
0,bc_initialization,wf_2023_2024,31.124073,5.018899,13.361110,16.693087,True,False,True,-26.105174,26.105174,3.331977,3.331977,True,True,relative_and_drawdown_gate_flip,block_training_until_official_replay_metric_al...
1,bc_initialization,wf_2022,66.727389,67.624396,14.997907,22.261989,True,False,True,0.897006,0.000000,7.264082,7.264082,False,True,max_drawdown_understated_gate_flip,block_training_until_official_replay_metric_al...


,candidate,fold,daily_path,rank,peak_date,trough_date,start,end,rows,max_dd,max_dd_fraction,return,avg_pos,peak_nav,trough_nav,interval_trades
0,bc_initialization,wf_2022,runtime/model_selection/card/short_volshock_bc...,1,2022-01-03,2022-11-03,2022-01-03,2022-11-03,212,22.2620,0.222620,-22.2620,0.086226,1.000000,0.777380,27
1,bc_initialization,wf_2023_2024,runtime/model_selection/card/short_volshock_bc...,1,2024-03-07,2024-11-15,2024-03-07,2024-11-15,177,16.6931,0.166931,-16.6931,0.140472,1.250300,1.041586,28
2,bc_initialization,wf_2023_2024,runtime/model_selection/card/short_volshock_bc...,2,2024-01-25,2024-01-31,2024-01-25,2024-01-31,5,5.9209,0.059209,-5.9209,0.421099,1.078817,1.014941,1
3,bc_initialization,wf_2023_2024,runtime/model_selection/card/short_volshock_bc...,3,2024-02-09,2024-02-20,2024-02-09,2024-02-20,7,4.8551,0.048551,-4.8551,0.482089,1.137090,1.081884,1


In [2]:
init_qlib(QLIB_DATA_PATH)
feature_by_fold: dict[str, pd.DataFrame] = {}
coverage_rows = []
for fold, spec in WALK_FORWARD_WINDOWS.items():
    dataset = build_dataset(
        train_seg=spec["train"],
        valid_seg=spec["valid"],
        test_seg=spec["test"],
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    features = flatten_feature_frame(dataset.handler.feature_frame).droplevel("instrument").copy()
    features.index = pd.to_datetime(features.index)
    features = features.sort_index()
    feature_by_fold[fold] = features
    coverage_rows.append({
        "fold": fold,
        "feature_rows": int(len(features)),
        "feature_cols": int(len(features.columns)),
        "feature_start": str(features.index.min().date()),
        "feature_end": str(features.index.max().date()),
    })
coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(CARD / "feature_coverage.csv", index=False)
display(coverage)


[75627:MainThread](2026-05-27 01:53:15,870) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[75627:MainThread](2026-05-27 01:53:16,592) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[75627:MainThread](2026-05-27 01:53:16,593) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


[75627:MainThread](2026-05-27 01:53:18,790) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[75627:MainThread](2026-05-27 01:53:18,793) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[75627:MainThread](2026-05-27 01:53:18,798) INFO - qlib.timer - [log.py:127] - Time cost: 0.008s | Init data Done


[75627:MainThread](2026-05-27 01:53:21,203) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | Loading data Done


[75627:MainThread](2026-05-27 01:53:21,204) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[75627:MainThread](2026-05-27 01:53:21,206) INFO - qlib.timer - [log.py:127] - Time cost: 0.003s | Init data Done


,fold,feature_rows,feature_cols,feature_start,feature_end
0,wf_2022,2973,59,2011-03-11,2022-12-30
1,wf_2023_2024,3475,59,2011-03-11,2024-12-31


In [3]:
delta_frames = []
group_frames = []
fold_group_frames = []
daily_feature_manifest = []

for row in source_pk.to_dict("records"):
    fold = str(row["fold"])
    candidate = str(row["candidate"])
    daily_path = ROOT / str(row["daily_path"])
    daily = pd.read_csv(daily_path, parse_dates=["datetime"])
    diagnostic_daily = daily.rename(columns={"strategy_nav": "account"}).copy()
    features = feature_by_fold[fold].copy()
    joined = diagnostic_daily.set_index("datetime").join(features.reindex(pd.DatetimeIndex(diagnostic_daily["datetime"]))).reset_index()
    joined_path = CARD / f"{candidate}_{fold}_official_daily_with_features.csv"
    joined.to_csv(joined_path, index=False)
    daily_feature_manifest.append({"candidate": candidate, "fold": fold, "path": str(joined_path.relative_to(ROOT))})

    fold_intervals = source_intervals[
        source_intervals["fold"].astype(str).eq(fold)
        & source_intervals["candidate"].astype(str).eq(candidate)
    ].copy()
    deltas = compare_interval_feature_deltas(
        candidate_daily=joined,
        intervals=fold_intervals,
        baseline_daily=None,
        feature_cols=list(features.columns),
    )
    if deltas.empty:
        continue
    deltas.insert(0, "candidate", candidate)
    deltas.insert(1, "fold", fold)
    delta_frames.append(deltas)
    groups = summarize_interval_feature_groups(deltas, top_n=12)
    if not groups.empty:
        groups.insert(0, "candidate", candidate)
        groups.insert(1, "fold", fold)
        group_frames.append(groups)

feature_deltas = pd.concat(delta_frames, ignore_index=True) if delta_frames else pd.DataFrame()
feature_groups = pd.concat(group_frames, ignore_index=True) if group_frames else pd.DataFrame()
if not feature_deltas.empty:
    feature_deltas.to_csv(CARD / "official_interval_feature_deltas.csv", index=False)
else:
    pd.DataFrame().to_csv(CARD / "official_interval_feature_deltas.csv", index=False)
if not feature_groups.empty:
    feature_groups.to_csv(CARD / "official_interval_feature_groups.csv", index=False)
else:
    pd.DataFrame().to_csv(CARD / "official_interval_feature_groups.csv", index=False)
pd.DataFrame(daily_feature_manifest).to_csv(CARD / "daily_feature_manifest.csv", index=False)

display(feature_deltas.head(40))
display(feature_groups)


,candidate,fold,interval_rank,interval_start,interval_end,feature,interval_mean,reference_mean,delta,delta_z,interval_coverage,reference_coverage,candidate_avg_pos,baseline_avg_pos,position_delta,candidate_return,baseline_return,return_delta,abs_delta_z
0,bc_initialization,wf_2022,1,2022-01-03,2022-11-03,anchor_slope_180,-0.062754,-0.151561,0.088806,7.915318,1.0,1.0,0.086226,0.0,0.086226,-22.262,0.0,-22.262,7.915318
1,bc_initialization,wf_2022,1,2022-01-03,2022-11-03,vix_zscore,0.960588,-0.706139,1.666727,4.433638,1.0,1.0,0.086226,0.0,0.086226,-22.262,0.0,-22.262,4.433638
2,bc_initialization,wf_2022,1,2022-01-03,2022-11-03,regime_risk,0.668759,0.377113,0.291647,4.432143,1.0,1.0,0.086226,0.0,0.086226,-22.262,0.0,-22.262,4.432143
3,bc_initialization,wf_2022,1,2022-01-03,2022-11-03,vix_percentile_252,0.753276,0.277934,0.475341,4.182156,1.0,1.0,0.086226,0.0,0.086226,-22.262,0.0,-22.262,4.182156
4,bc_initialization,wf_2022,1,2022-01-03,2022-11-03,downside_semivol_21,0.940624,0.723265,0.217359,3.003058,1.0,1.0,0.086226,0.0,0.086226,-22.262,0.0,-22.262,3.003058
5,bc_initialization,wf_2022,1,2022-01-03,2022-11-03,vix_level,0.524428,0.447889,0.076539,2.386763,1.0,1.0,0.086226,0.0,0.086226,-22.262,0.0,-22.262,2.386763
6,bc_initialization,wf_2022,1,2022-01-03,2022-11-03,realized_skew_21,-0.001560,0.875496,-0.877056,-2.218620,1.0,1.0,0.086226,0.0,0.086226,-22.262,0.0,-22.262,2.218620
7,bc_initialization,wf_2022,1,2022-01-03,2022-11-03,realized_kurtosis_21,-0.420251,1.538688,-1.958939,-2.002171,1.0,1.0,0.086226,0.0,0.086226,-22.262,0.0,-22.262,2.002171
8,bc_initialization,wf_2022,1,2022-01-03,2022-11-03,cum_drop_30,-0.328115,-0.135349,-0.192766,-1.982397,1.0,1.0,0.086226,0.0,0.086226,-22.262,0.0,-22.262,1.982397
9,bc_initialization,wf_2022,1,2022-01-03,2022-11-03,below_ma63_days,0.568733,0.092732,0.476001,1.909413,1.0,1.0,0.086226,0.0,0.086226,-22.262,0.0,-22.262,1.909413


,candidate,fold,feature_group,feature_count,mean_abs_delta_z,max_abs_delta_z,mean_delta_z,mean_position_delta,mean_return_delta,top_features
0,bc_initialization,wf_2022,volatility_or_stress,20,1.579191,4.433638,0.909752,0.086226,-22.262000,"vix_zscore,regime_risk,vix_percentile_252,down..."
1,bc_initialization,wf_2022,other,14,1.550559,7.915318,-0.105912,0.086226,-22.262000,"anchor_slope_180,realized_skew_21,realized_kur..."
2,bc_initialization,wf_2022,price_level,2,0.973109,1.909413,0.973109,0.086226,-22.262000,"below_ma63_days,days_since_high_63"
3,bc_initialization,wf_2022,relative_strength_spread,10,0.717675,1.272277,0.017689,0.086226,-22.262000,"tracking_error_21,relative_strength_21,rolling..."
4,bc_initialization,wf_2022,trend_or_momentum,13,0.699877,1.592115,-0.548583,0.086226,-22.262000,"trend_slope_21,mom_21,trend_gap_63,trend_gap_21"
5,bc_initialization,wf_2023_2024,volatility_or_stress,20,0.877632,5.217874,0.184357,0.347887,-9.156367,"vol_of_vol_21,vol_coef_21,realized_vol_21,real..."
6,bc_initialization,wf_2023_2024,price_level,2,0.798944,1.230984,-0.477904,0.347887,-9.156367,"days_since_high_63,days_since_high_63,days_sin..."
7,bc_initialization,wf_2023_2024,trend_or_momentum,13,0.646555,1.995240,0.359870,0.347887,-9.156367,"mom_63,trend_slope_63,rsi_14,trend_gap_21"
8,bc_initialization,wf_2023_2024,other,14,0.608838,2.357670,0.177789,0.347887,-9.156367,"decay_63,expectancy_proxy_21,sharpe_proxy_21,n..."
9,bc_initialization,wf_2023_2024,relative_strength_spread,10,0.578729,2.327250,-0.185938,0.347887,-9.156367,"tracking_error_vol_21,tracking_error_vol_21,tr..."


In [4]:
if feature_deltas.empty:
    feature_group_summary = pd.DataFrame()
else:
    feature_group_summary = summarize_interval_feature_groups(feature_deltas, top_n=20)
    # Add fold/candidate coverage to the global group ranking.
    coverage = []
    for group in feature_group_summary["feature_group"].tolist():
        sub = feature_deltas[feature_deltas["feature"].map(lambda x: summarize_interval_feature_groups(pd.DataFrame({"feature": [x], "delta_z": [1.0]})).iloc[0]["feature_group"] if x else "other").eq(group)]
        coverage.append({
            "feature_group": group,
            "fold_count": int(sub["fold"].nunique()) if "fold" in sub.columns else 0,
            "candidate_count": int(sub["candidate"].nunique()) if "candidate" in sub.columns else 0,
        })
    coverage_df = pd.DataFrame(coverage)
    feature_group_summary = feature_group_summary.merge(coverage_df, on="feature_group", how="left")

feature_group_summary.to_csv(CARD / "feature_group_summary.csv", index=False)

top_features = feature_deltas.sort_values(["abs_delta_z", "feature"], ascending=[False, True]).head(30) if not feature_deltas.empty else pd.DataFrame()
top_features.to_csv(CARD / "top_interval_features.csv", index=False)

display(feature_group_summary)
display(top_features)


,feature_group,feature_count,mean_abs_delta_z,max_abs_delta_z,mean_delta_z,mean_position_delta,mean_return_delta,top_features,fold_count,candidate_count
0,volatility_or_stress,20,1.053022,5.217874,0.365706,0.282472,-12.432775,"vol_of_vol_21,vix_zscore,regime_risk,vix_perce...",2,1
1,other,14,0.844268,7.915318,0.106864,0.282471,-12.432775,"anchor_slope_180,decay_63,realized_skew_21,rea...",2,1
2,price_level,2,0.842485,1.909413,-0.115151,0.282471,-12.432775,"below_ma63_days,days_since_high_63,days_since_...",2,1
3,trend_or_momentum,13,0.659885,1.995240,0.132757,0.282472,-12.432775,"mom_63,trend_slope_63,trend_slope_21,rsi_14",2,1
4,relative_strength_spread,10,0.613465,2.327250,-0.135031,0.282471,-12.432775,"tracking_error_vol_21,tracking_error_vol_21,tr...",2,1


,candidate,fold,interval_rank,interval_start,interval_end,feature,interval_mean,reference_mean,delta,delta_z,interval_coverage,reference_coverage,candidate_avg_pos,baseline_avg_pos,position_delta,candidate_return,baseline_return,return_delta,abs_delta_z
0,bc_initialization,wf_2022,1,2022-01-03,2022-11-03,anchor_slope_180,-0.062754,-0.151561,0.088806,7.915318,1.0,1.0,0.086226,0.0,0.086226,-22.2620,0.0,-22.2620,7.915318
59,bc_initialization,wf_2023_2024,1,2024-03-07,2024-11-15,vol_of_vol_21,0.134967,0.065245,0.069722,5.217874,1.0,1.0,0.140472,0.0,0.140472,-16.6931,0.0,-16.6931,5.217874
1,bc_initialization,wf_2022,1,2022-01-03,2022-11-03,vix_zscore,0.960588,-0.706139,1.666727,4.433638,1.0,1.0,0.086226,0.0,0.086226,-22.2620,0.0,-22.2620,4.433638
2,bc_initialization,wf_2022,1,2022-01-03,2022-11-03,regime_risk,0.668759,0.377113,0.291647,4.432143,1.0,1.0,0.086226,0.0,0.086226,-22.2620,0.0,-22.2620,4.432143
3,bc_initialization,wf_2022,1,2022-01-03,2022-11-03,vix_percentile_252,0.753276,0.277934,0.475341,4.182156,1.0,1.0,0.086226,0.0,0.086226,-22.2620,0.0,-22.2620,4.182156
60,bc_initialization,wf_2023_2024,1,2024-03-07,2024-11-15,vol_coef_21,1.280448,0.999213,0.281235,3.629545,1.0,1.0,0.140472,0.0,0.140472,-16.6931,0.0,-16.6931,3.629545
61,bc_initialization,wf_2023_2024,1,2024-03-07,2024-11-15,realized_vol_21,1.041541,0.834174,0.207367,3.337241,1.0,1.0,0.140472,0.0,0.140472,-16.6931,0.0,-16.6931,3.337241
4,bc_initialization,wf_2022,1,2022-01-03,2022-11-03,downside_semivol_21,0.940624,0.723265,0.217359,3.003058,1.0,1.0,0.086226,0.0,0.086226,-22.2620,0.0,-22.2620,3.003058
62,bc_initialization,wf_2023_2024,1,2024-03-07,2024-11-15,realized_vol_63,1.051178,0.835121,0.216057,2.726562,1.0,1.0,0.140472,0.0,0.140472,-16.6931,0.0,-16.6931,2.726562
5,bc_initialization,wf_2022,1,2022-01-03,2022-11-03,vix_level,0.524428,0.447889,0.076539,2.386763,1.0,1.0,0.086226,0.0,0.086226,-22.2620,0.0,-22.2620,2.386763


In [5]:
top_group = str(feature_group_summary.iloc[0]["feature_group"]) if not feature_group_summary.empty else "none"
if top_group in {"volatility_or_stress", "relative_strength_spread"}:
    decision = "official_replay_bad_interval_state_feature_audit_routes_to_risk_state_selection"
    next_mechanism = "build official-replay-gated risk-state checkpoint or target screen before PPO"
elif top_group == "trend_or_momentum":
    decision = "official_replay_bad_interval_state_feature_audit_routes_to_trend_participation_selection"
    next_mechanism = "screen trend-participation objective under official replay gates before PPO"
else:
    decision = "official_replay_bad_interval_state_feature_audit_no_clear_group"
    next_mechanism = "aggregate with additional official replay failures before training"

conclusion = []
if not feature_groups.empty:
    for row in feature_groups.sort_values(["fold", "mean_abs_delta_z"], ascending=[True, False]).groupby("fold").head(3).to_dict("records"):
        conclusion.append(
            f"{row['fold']}: top group {row['feature_group']} mean_abs_delta_z={row['mean_abs_delta_z']:.2f}, "
            f"features={row['top_features']}."
        )
if not top_features.empty:
    for row in top_features.head(8).to_dict("records"):
        conclusion.append(
            f"top feature {row['fold']} {row['feature']}: delta_z={row['delta_z']:.2f}, "
            f"interval={row['interval_start']}..{row['interval_end']}, avg_pos={row['candidate_avg_pos']:.3f}."
        )
conclusion.append("This is feature attribution only; high delta_z features are hypotheses for official-replay-gated selection, not promoted factors.")
conclusion.append(f"Next mechanism: {next_mechanism}.")

selection = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": decision,
    "role": "diagnose",
    "changed_layer": ["model_selection", "feature_attribution", "bad_interval_screen"],
    "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
    "direct_comparison_baseline": "metric_alignment_gate_flip_audit_v1",
    "profile": PROFILE_NAME,
    "source_metric_decision": metric_selection.get("decision"),
    "target_relative_return_pp": TARGET_RELATIVE_RETURN,
    "target_max_dd_pct": TARGET_MAX_DD,
    "top_feature_group": top_group,
    "promotion_allowed": False,
    "actor_retention": "no actor retained; this card only routes feature hypotheses",
    "next_mechanism": next_mechanism,
    "external_sources": EXTERNAL_SOURCES,
    "sidecars": [
        "feature_coverage.csv",
        "daily_feature_manifest.csv",
        "official_interval_feature_deltas.csv",
        "official_interval_feature_groups.csv",
        "feature_group_summary.csv",
        "top_interval_features.csv",
        "selection.json",
        "summary.json",
        "*_official_daily_with_features.csv",
    ],
}
summary = {
    "experiment": EXPERIMENT,
    "created_at": selection["created_at"],
    "decision": decision,
    "promotion_allowed": False,
    "top_feature_group": top_group,
    "feature_group_summary": feature_group_summary.to_dict("records") if not feature_group_summary.empty else [],
    "top_interval_features": top_features.head(30).to_dict("records") if not top_features.empty else [],
    "source_audit": source_audit.to_dict("records"),
    "conclusion": conclusion,
    "next_mechanism": next_mechanism,
    "external_sources": EXTERNAL_SOURCES,
    "sidecars": selection["sidecars"],
}
write_json(CARD / "selection.json", selection)
write_json(CARD / "summary.json", summary)
print(json.dumps(json_safe(selection), indent=2, allow_nan=False))


{
  "experiment": "official_replay_bad_interval_feature_audit_v1",
  "created_at": "2026-05-26T17:53:24.629143+00:00",
  "decision": "official_replay_bad_interval_state_feature_audit_routes_to_risk_state_selection",
  "role": "diagnose",
  "changed_layer": [
    "model_selection",
    "feature_attribution",
    "bad_interval_screen"
  ],
  "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
  "direct_comparison_baseline": "metric_alignment_gate_flip_audit_v1",
  "profile": "sparse_release_bc_ppo_screen_v1",
  "source_metric_decision": "metric_alignment_gate_flip_blocks_training",
  "target_relative_return_pp": 30.0,
  "target_max_dd_pct": 15.0,
  "top_feature_group": "volatility_or_stress",
  "promotion_allowed": false,
  "actor_retention": "no actor retained; this card only routes feature hypotheses",
  "next_mechanism": "build official-replay-gated risk-state checkpoint or target screen before PPO",
  "external_sources": [
    "https://qlib.readthedocs.io/en/latest/compone

## Conclusion

The notebook output above is the single-run conclusion. It routes the next experiment from official replay failures only; it does not promote any factor or model.

In [6]:
summary = json.loads((CARD / "summary.json").read_text(encoding="utf-8"))
lines = ["# Runtime Conclusion", ""]
lines.append(f"- decision: `{summary['decision']}`")
lines.append(f"- top_feature_group: `{summary['top_feature_group']}`")
lines.append(f"- next: `{summary['next_mechanism']}`")
lines.append("- promotion_allowed: `False`")
lines.append("")
for item in summary.get("conclusion", []):
    lines.append(f"- {item}")
display(Markdown("\n".join(lines)))


# Runtime Conclusion

- decision: `official_replay_bad_interval_state_feature_audit_routes_to_risk_state_selection`
- top_feature_group: `volatility_or_stress`
- next: `build official-replay-gated risk-state checkpoint or target screen before PPO`
- promotion_allowed: `False`

- wf_2022: top group volatility_or_stress mean_abs_delta_z=1.58, features=vix_zscore,regime_risk,vix_percentile_252,downside_semivol_21.
- wf_2022: top group other mean_abs_delta_z=1.55, features=anchor_slope_180,realized_skew_21,realized_kurtosis_21,cum_drop_30.
- wf_2022: top group price_level mean_abs_delta_z=0.97, features=below_ma63_days,days_since_high_63.
- wf_2023_2024: top group volatility_or_stress mean_abs_delta_z=0.88, features=vol_of_vol_21,vol_coef_21,realized_vol_21,realized_vol_63.
- wf_2023_2024: top group price_level mean_abs_delta_z=0.80, features=days_since_high_63,days_since_high_63,days_since_high_63,below_ma63_days.
- wf_2023_2024: top group trend_or_momentum mean_abs_delta_z=0.65, features=mom_63,trend_slope_63,rsi_14,trend_gap_21.
- top feature wf_2022 anchor_slope_180: delta_z=7.92, interval=2022-01-03..2022-11-03, avg_pos=0.086.
- top feature wf_2023_2024 vol_of_vol_21: delta_z=5.22, interval=2024-03-07..2024-11-15, avg_pos=0.140.
- top feature wf_2022 vix_zscore: delta_z=4.43, interval=2022-01-03..2022-11-03, avg_pos=0.086.
- top feature wf_2022 regime_risk: delta_z=4.43, interval=2022-01-03..2022-11-03, avg_pos=0.086.
- top feature wf_2022 vix_percentile_252: delta_z=4.18, interval=2022-01-03..2022-11-03, avg_pos=0.086.
- top feature wf_2023_2024 vol_coef_21: delta_z=3.63, interval=2024-03-07..2024-11-15, avg_pos=0.140.
- top feature wf_2023_2024 realized_vol_21: delta_z=3.34, interval=2024-03-07..2024-11-15, avg_pos=0.140.
- top feature wf_2022 downside_semivol_21: delta_z=3.00, interval=2022-01-03..2022-11-03, avg_pos=0.086.
- This is feature attribution only; high delta_z features are hypotheses for official-replay-gated selection, not promoted factors.
- Next mechanism: build official-replay-gated risk-state checkpoint or target screen before PPO.